In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg import Connection
from psycopg.rows import dict_row

# 从 Notebook 目录或仓库根目录定位 agent-service。
cwd = Path.cwd()
service_root = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "agent-service")
        if (candidate / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("找不到 agent-service，请从项目目录打开 Notebook。")
if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))

load_dotenv(service_root / ".env")
from app.core.config import settings

# Windows Jupyter 的 ProactorEventLoop 与 Psycopg 异步连接不兼容。
# 连接保留给后续单元使用，重跑本单元时关闭旧连接。
if "checkpoint_connection" in globals():
    checkpoint_connection.close()

checkpoint_connection = Connection.connect(
    settings.db.checkpoint_url,
    connect_timeout=10,
    autocommit=True,
    prepare_threshold=0,
    row_factory=dict_row,
)
try:
    checkpointer = PostgresSaver(checkpoint_connection)
    checkpointer.setup()
except BaseException:
    checkpoint_connection.close()
    raise

print("PostgreSQL Checkpointer 表已就绪。")

PostgreSQL Checkpointer 表已就绪。


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具


# 4.创建Agent，绑定模型和工具，Checkpointer
agent = create_agent(
    model = model,
    tools=[],
    checkpointer=checkpointer
)

In [ ]:
# 5.调用Agent，要能查询记忆，必须携带thread_id
config = {"configurable": {"thread_id": "thread_01"}}

response = agent.invoke(
    {"messages": [HumanMessage("12个苹果分给4个人，每人分几个？")]},
    config=config,
)
print(response['messages'][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage("如果是3个人呢？")]},
    config=config,
)
print(response['messages'][-1].content)

In [ ]:
# 查询会话历史
state_snapshot = agent.get_state(config)
print(state_snapshot)

In [ ]:
{"role": "", "content": ""}
# SystemMessage -> system
# HumanMessage -> user
# AIMessage -> assistant
# ToolMessage -> tool


In [ ]:
from langchain.messages import HumanMessage, AIMessage

messages = []

for message in state_snapshot.values['messages']:
    # 判断类型，决定role
    if isinstance(message, HumanMessage):
        role = "user"
    elif isinstance(message, AIMessage):
        role = "assistant"
    else:
        continue

    messages.append({"role": role, "content": message.text})

print(messages)

In [ ]:
checkpointer.delete_thread("7d63e96f-b8c2-4781-b9cc-faf7eefa3ea5")

In [ ]:
state_snapshot = agent.get_state(config)
print(state_snapshot)

In [ ]:
config = {"configurable": {"thread_id": "de55b937-1265-4122-961a-c22accc07ab3"}}
state_snapshot = agent.get_state(config)
print(state_snapshot)